# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.

   Dataset **Diabetes, Hypertension and Stroke Prediction**
   
   Sumber: https://www.kaggle.com/datasets/prosperchuks/health-dataset
   
   Deskripsi:
   
   Dataset ini berisi data hasil survei dari BRFSS 2015 (Behavioral Risk Factor Surveillance System) yang telah dibersihkan, ditingkatkan (augmented), dan diselaraskan agar memiliki kelas yang seimbang (balanced classes). Dataset ini sangat cocok digunakan untuk riset, pembelajaran machine learning, maupun pembuatan aplikasi klasifikasi prediksi risiko kesehatan.
   Secara keseluruhan, dataset ini terbagi menjadi 3 file CSV utama:
   1. diabetes_data.csv (5.29 MB)
   2. hypertension_data.csv
   3. stroke_data.csv

   Pada eksperimen ini, dataset yang digunakan adalah **diabetes_data.csv**

   


# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
#Type your code here
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [2]:
#Type your code here
import kagglehub
import os

# Download dataset
path = kagglehub.dataset_download("prosperchuks/health-dataset")
print(path)

# Melihat isi folder
os.listdir(path)

data = pd.read_csv(
    os.path.join(path,"diabetes_data.csv")
)


100%|██████████| 1.00M/1.00M [00:00<00:00, 99.5MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/prosperchuks/health-dataset/versions/5


# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [3]:
print("Dataset Info:")
data.info()

print("\nDescriptive Statistics:")
print(data.describe())

print("\nMissing Values:")
print(data.isnull().sum())

print("\nValue Counts for Categorical Features:")
# Identify categorical columns (excluding numerical/ordinal already partly visualized or covered by describe)
categorical_cols = [
    'Sex', 'HighChol', 'CholCheck', 'Smoker', 'HeartDiseaseorAttack',
    'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'GenHlth',
    'DiffWalk', 'Stroke', 'HighBP', 'Diabetes'
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(data[col].value_counts())
    print("Percentage:")
    print(data[col].value_counts(normalize=True).mul(100).map('{:.2f}%'.format))

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70692 entries, 0 to 70691
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   70692 non-null  float64
 1   Sex                   70692 non-null  float64
 2   HighChol              70692 non-null  float64
 3   CholCheck             70692 non-null  float64
 4   BMI                   70692 non-null  float64
 5   Smoker                70692 non-null  float64
 6   HeartDiseaseorAttack  70692 non-null  float64
 7   PhysActivity          70692 non-null  float64
 8   Fruits                70692 non-null  float64
 9   Veggies               70692 non-null  float64
 10  HvyAlcoholConsump     70692 non-null  float64
 11  GenHlth               70692 non-null  float64
 12  MentHlth              70692 non-null  float64
 13  PhysHlth              70692 non-null  float64
 14  DiffWalk              70692 non-null  float64
 15  Strok

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [6]:
target = "Diabetes"

X = data.drop(columns=[target])
y = data[target]

num_cols = X.select_dtypes(include=np.number).columns

cat_cols = X.select_dtypes(exclude=np.number).columns

numeric_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num",numeric_pipeline,num_cols),
    ("cat",categorical_pipeline,cat_cols)
])

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Shape of processed training data:", X_train_processed.shape)
print("Shape of processed testing data:", X_test_processed.shape)

Shape of processed training data: (56553, 17)
Shape of processed testing data: (14139, 17)


In [7]:
# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Convert processed arrays back to DataFrames
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

# Reset index for y_train and y_test to ensure proper concatenation
y_train_reset = y_train.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

# Combine X_train_processed_df with y_train
train_df = pd.concat([X_train_processed_df, y_train_reset], axis=1)

# Combine X_test_processed_df with y_test
test_df = pd.concat([X_test_processed_df, y_test_reset], axis=1)

# Concatenate train and test dataframes
preprocessed_data = pd.concat([train_df, test_df], ignore_index=True)

# Save the preprocessed data to a CSV file
output_filename = "preprocessed_diabetes_data.csv"
preprocessed_data.to_csv(output_filename, index=False)

print(f"Preprocessed data saved to {output_filename}")
print("Shape of the final preprocessed dataset:", preprocessed_data.shape)
print("First 5 rows of the preprocessed dataset:")
print(preprocessed_data.head())

Preprocessed data saved to preprocessed_diabetes_data.csv
Shape of the final preprocessed dataset: (70692, 18)
First 5 rows of the preprocessed dataset:
   num__Age  num__Sex  num__HighChol  num__CholCheck  num__BMI  num__Smoker  \
0  0.496401  1.090144      -1.052422        0.158387 -0.119897     1.050409   
1 -0.555625 -0.917310       0.950189        0.158387 -0.822423     1.050409   
2  0.847076 -0.917310      -1.052422        0.158387  1.706673     1.050409   
3  0.496401 -0.917310       0.950189        0.158387  0.020609    -0.952010   
4 -0.906300  1.090144      -1.052422        0.158387  0.723136     1.050409   

   num__HeartDiseaseorAttack  num__PhysActivity  num__Fruits  num__Veggies  \
0                  -0.415678           0.650525     0.795861      0.518296   
1                  -0.415678           0.650525    -1.256502      0.518296   
2                  -0.415678           0.650525     0.795861      0.518296   
3                   2.405711           0.650525     0.795861